# Drift Detection Analysis
This notebook performs drift detection analysis on old_data.csv and new_data.csv using the monitoring package.

In [2]:
# Import required libraries
import sys
import os
from pathlib import Path

# Add the monitoring_package to the Python path
# Since we're in the 'data' folder, go up one level to reach the parent
parent_dir = Path(os.getcwd()).parent
monitoring_path = parent_dir / 'monitoring_package'
sys.path.insert(0, str(monitoring_path))

print(f"Added to path: {monitoring_path}")
print(f"Path exists: {monitoring_path.exists()}")

Added to path: c:\Users\G705728\OneDrive - General Mills\Desktop\Drift detection POC\Drift-Detection-and-RCA\monitoring_package
Path exists: True


In [3]:
# Import monitoring package modules
import monitoring.data_drift as data_drift
from monitoring.data_drift import run_drift_tests
from monitoring.drift_report import generate_html_report
import pandas as pd
import json

print("Successfully imported monitoring modules!")

Successfully imported monitoring modules!


In [25]:
# Load the data files
previous = pd.read_csv('old_data.csv')
current = pd.read_csv('new_data.csv')

print(f"Previous data shape: {previous.shape}")
print(f"Current data shape: {current.shape}")
print(f"\nPrevious data columns: {list(previous.columns)}")
print(f"Current data columns: {list(current.columns)}")

Previous data shape: (31, 13)
Current data shape: (31, 13)

Previous data columns: ['date', 'VendorID', 'passenger_count', 'trip_distance', 'RateCodeID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']
Current data columns: ['date', 'VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


In [7]:
# Check if columns match between datasets
common_columns = set(previous.columns) & set(current.columns)
only_in_previous = set(previous.columns) - set(current.columns)
only_in_current = set(current.columns) - set(previous.columns)

print(f"Common columns ({len(common_columns)}): {sorted(common_columns)}")
if only_in_previous:
    print(f"\nOnly in previous: {sorted(only_in_previous)}")
if only_in_current:
    print(f"\nOnly in current: {sorted(only_in_current)}")

Common columns (12): ['VendorID', 'date', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'tip_amount', 'tolls_amount', 'total_amount', 'trip_distance']

Only in previous: ['RateCodeID']

Only in current: ['RatecodeID']


In [13]:
# Exclude problematic columns
columns_to_exclude = ['RateCodeID']

# Remove excluded columns from both datasets if they exist
for col in columns_to_exclude:
    if col in previous.columns:
        previous = previous.drop(columns=[col])
        print(f"Removed '{col}' from previous dataset")
    if col in current.columns:
        current = current.drop(columns=[col])
        print(f"Removed '{col}' from current dataset")

print(f"\nPrevious data shape after exclusion: {previous.shape}")
print(f"Current data shape after exclusion: {current.shape}")

Removed 'RateCodeID' from previous dataset

Previous data shape after exclusion: (31, 12)
Current data shape after exclusion: (31, 13)


In [26]:
# Run drift detection tests only on common columns
# Exclude RateCodeID and any other problematic columns
result = run_drift_tests(
    previous, 
    current,
    # columns_to_exclude=['RateCodeID'],
    columns_to_test=['total_amount']
        # Explicitly exclude problematic columns
)

print("Drift detection completed successfully!")

ERROR:monitoring.data_drift:Failed to generate HTML report: 'dict object' has no attribute 'RateCodeID'


[2025-12-11 13:46:46] [data_drift] Starting run_drift_tests...
[2025-12-11 13:46:46] [data_drift] All columns processed in 0.008 seconds
[2025-12-11 13:46:46] [data_drift] drift_structured_log.json written in 0.004 seconds
[2025-12-11 13:46:46] [data_drift] [drift_report] Generating HTML report...
[2025-12-11 13:46:46] [drift_report] Starting HTML report generation...
[2025-12-11 13:46:46] [drift_report] Template loaded.
[2025-12-11 13:46:46] [drift_report] Result sanitization took 0.000 seconds
[2025-12-11 13:46:46] [data_drift] Program terminated. Total run_drift_tests time: 0.136 seconds
Drift detection completed successfully!


In [29]:
# # Print results in a readable format
# print(json.dumps(result, indent=2, default=str))
import json
print(json.dumps(result, indent=2, default=str))

# Save results
with open('drift_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)

# Generate HTML report

{
  "is_drift": true,
  "details": {
    "total_amount": {
      "tests": {
        "ks": {
          "result": {
            "statistic": 0.41935483870967744,
            "p_value": 0.007945718302533482,
            "drift": true
          },
          "drift": true,
          "weight": 0.3333333333333333,
          "threshold": 0.05
        },
        "wasserstein": {
          "result": {
            "wasserstein_distance": 710224.472580645,
            "wasserstein_distance_norm": 0.5271045152426479,
            "drift": true
          },
          "drift": true,
          "weight": 0.3333333333333333,
          "threshold": 0.1
        },
        "psi": {
          "result": {
            "psi": 8.941493045898543,
            "drift": true
          },
          "drift": true,
          "weight": 0.3333333333333333,
          "threshold": 0.2
        }
      },
      "is_drift": true
    }
  },
  "data_summary": {
    "columns": [
      "date",
      "VendorID",
      "passenger_c

In [23]:
# Save results to JSON file
with open('drift_structured_log.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)

print("Results saved to drift_results.json")

Results saved to drift_results.json


In [17]:
# Generate HTML report
generate_html_report(result, output_path='drift_report.html')

print("HTML report generated: drift_report.html")

[2025-12-11 12:44:10] [drift_report] Starting HTML report generation...
[2025-12-11 12:44:10] [drift_report] Template loaded.
[2025-12-11 12:44:10] [drift_report] Result sanitization took 0.000 seconds
[2025-12-11 12:44:10] [drift_report] Template rendering took 0.001 seconds
[2025-12-11 12:44:10] [drift_report] File write took 0.003 seconds
[2025-12-11 12:44:10] [drift_report] HTML report saved to drift_report.html
[2025-12-11 12:44:10] [drift_report] Total drift_report.py time: 0.027 seconds
[2025-12-11 12:44:10] [drift_report] Program terminated. Total time: 0.027 seconds
HTML report generated: drift_report.html
